# Sheffield Scrollie — Segmentation Viewer

Interactive slice-by-slice viewer for algorithms run on the **Sheffield** dataset.

- **Images**: `sheffeld/20440164/Aug_N.dcm` (multi-frame greyscale DICOM)
- **Ground truth**: `sheffeld/20440203/Aug_N_segmentations.dcm` (37 bilateral muscle labels)

**How to use:**
1. Run all cells.
2. Pick up to **two algorithms** from the dropdowns.
3. Pick a sample (`Aug_N`) from the Stack dropdown.
4. Toggle **Show GT** for ground-truth overlay.
5. Drag the slice slider.

In [ ]:
import glob
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import SimpleITK as sitk
import pydicom
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, VBox, HBox
from IPython.display import display

In [ ]:
# ── Sheffield GT label map — 37 bilateral muscles (labels confirmed from column-compare notebooks) ──
SHEFFIELD_GT_LABELS = {
    1:  'adductor_brevis',
    2:  'adductor_longus',
    3:  'adductor_magnus',
    4:  'biceps_femoris_short',
    5:  'biceps_femoris_long',
    6:  'label_6',
    7:  'label_7',
    8:  'label_8',
    9:  'label_9',
    10: 'label_10',
    11: 'label_11',
    12: 'label_12',
    13: 'label_13',
    14: 'label_14',
    15: 'label_15',
    16: 'gracilis',
    17: 'label_17',
    18: 'label_18',
    19: 'label_19',
    20: 'label_20',
    21: 'label_21',
    22: 'label_22',
    23: 'label_23',
    24: 'label_24',
    25: 'label_25',
    26: 'label_26',
    27: 'rectus_femoris',
    28: 'sartorius',
    29: 'semimembranosus',
    30: 'semitendinosus',
    31: 'label_31',
    32: 'label_32',
    33: 'label_33',
    34: 'label_34',
    35: 'vastus_intermedius',
    36: 'vastus_lateralis',
    37: 'vastus_medialis',
}

# ── Per-algorithm NIfTI label maps ─────────────────────────────────────────────
MM_WB_LABELS = {
    7101: 'VastusLat_L',      7102: 'VastusLat_R',
    7111: 'VastusInt_L',      7112: 'VastusInt_R',
    7121: 'VastusMed_L',      7122: 'VastusMed_R',
    7131: 'RectusFem_L',      7132: 'RectusFem_R',
    7141: 'Sartorius_L',      7142: 'Sartorius_R',
    7151: 'Gracilis_L',       7152: 'Gracilis_R',
    7161: 'Semimem_L',        7162: 'Semimem_R',
    7171: 'Semiten_L',        7172: 'Semiten_R',
    7181: 'BicepsFemL_L',     7182: 'BicepsFemL_R',
    7201: 'AdductorMag_L',    7202: 'AdductorMag_R',
    7221: 'AdductorBrev_L',   7222: 'AdductorBrev_R',
}

MM_THIGH_LABELS = {
    1:  'VastusLat_L',      2:  'VastusLat_R',
    3:  'VastusInt_L',      4:  'VastusInt_R',
    5:  'VastusMed_L',      6:  'VastusMed_R',
    7:  'RectusFem_L',      8:  'RectusFem_R',
    9:  'Sartorius_L',      10: 'Sartorius_R',
    11: 'Gracilis_L',       12: 'Gracilis_R',
    13: 'Semimem_L',        14: 'Semimem_R',
    15: 'Semiten_L',        16: 'Semiten_R',
    17: 'BicepsFemL_L',     18: 'BicepsFemL_R',
    19: 'BicepsFemS_L',     20: 'BicepsFemS_R',
    21: 'AddMag_L',         22: 'AddMag_R',
    23: 'AddLong_L',        24: 'AddLong_R',
    25: 'AddBrev_L',        26: 'AddBrev_R',
    27: 'Femur_L',          28: 'Femur_R',
}

HIRRIRIRIIR_LABELS = {
    1:  'Sartorius',     2:  'RectusFem',
    3:  'VastusLat',     4:  'VastusInt',
    5:  'VastusMed',     6:  'AddMag',
    7:  'Gracilis',      8:  'BicepsFemL',
    9:  'Semiten',       10: 'Semimem',
    11: 'BicepsFemS',
}

MUSEG_LABELS = {
    1:  'VastusLat',    2:  'VastusInt',
    3:  'VastusMed',    4:  'RectusFem',
    5:  'Sartorius',    6:  'Gracilis',
    7:  'Semimem',      8:  'Semiten',
    9:  'BicepsFemL',   10: 'BicepsFemS',
    11: 'AddMag',       12: 'AddLong',
    13: 'AddBrev',
}

print('Label maps defined.')

In [ ]:
# ── Configuration — edit / extend this cell ────────────────────────────────────
EVAL_DIR = r'C:\Projects\dissector\eval_notebooks'
IMG_DIR  = os.path.join(EVAL_DIR, 'sheffeld', '20440164')
GT_DIR   = os.path.join(EVAL_DIR, 'sheffeld', '20440203')

ALGORITHMS = {
    'MuscleMap WB': {
        'seg_dir': os.path.join(EVAL_DIR, 'muscle_map_wb', 'sheffield_segs'),
        'glob':    'Aug_*_dseg.nii.gz',
        'fmt':     'nifti',
        'label_map': MM_WB_LABELS,
    },
    'MuscleMap Thigh': {
        'seg_dir': os.path.join(EVAL_DIR, 'muscle_map_thigh', 'sheffield_segs'),
        'glob':    'Aug_*_dseg.nii.gz',
        'fmt':     'nifti',
        'label_map': MM_THIGH_LABELS,
    },
    'Hirriririir': {
        'seg_dir': os.path.join(EVAL_DIR, 'multimodal-multiethnic', 'sheffield_segs'),
        'glob':    'Aug_*_thigh_seg.nii.gz',
        'fmt':     'nifti',
        'label_map': HIRRIRIRIIR_LABELS,
    },
    'MuSeg': {
        'seg_dir': os.path.join(EVAL_DIR, 'museg', 'sheffield_segs'),
        'glob':    'Aug_*_dseg.nii.gz',
        'fmt':     'nifti',
        'label_map': MUSEG_LABELS,
    },
    'Dafne': {
        'seg_dir': os.path.join(EVAL_DIR, 'dafne', 'sheffield_segs'),
        'glob':    'Aug_*/*_dafne_thigh.npz',
        'fmt':     'npz',
        'label_map': None,
    },
    'MedCLIP-SAMv2': {
        'seg_dir': os.path.join(EVAL_DIR, 'medclipsamv2', 'sheffield_segs'),
        'glob':    'Aug_*_medclipsamv2.npz',
        'fmt':     'npz',
        'label_map': None,
    },
    'MedCLIP-SAMv2 Text+Boxes': {
        'seg_dir': os.path.join(EVAL_DIR, 'medclipsamv2textboxes', 'sheffield_segs'),
        'glob':    'Aug_*_mcsam2textboxes.npz',
        'fmt':     'npz',
        'label_map': None,
    },
    'MedSegDiff': {
        'seg_dir': os.path.join(EVAL_DIR, 'medsegdiff', 'sheffield_segs'),
        'glob':    'Aug_*_seg.npz',
        'fmt':     'npz',
        'label_map': None,
    },
    'SLM-SAM2': {
        'seg_dir': os.path.join(EVAL_DIR, 'muscle_map_wb+slmsam', 'sheffield_segs'),
        'glob':    'Aug_*_slmsam2.npz',
        'fmt':     'npz',
        'label_map': None,
    },
}

def _count(cfg):
    return len(glob.glob(os.path.join(cfg['seg_dir'], cfg['glob'])))

AVAILABLE = {
    name: cfg for name, cfg in ALGORITHMS.items()
    if os.path.isdir(cfg['seg_dir']) and _count(cfg) > 0
}

print(f'Available algorithms ({len(AVAILABLE)}/{len(ALGORITHMS)}):')
for name, cfg in AVAILABLE.items():
    print(f'  {name}: {_count(cfg)} files')

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────────

def load_sheffield_image(idx):
    """Load Aug_N.dcm greyscale DICOM → normalised (D,H,W) float32."""
    path = os.path.join(IMG_DIR, f'Aug_{idx}.dcm')
    ds   = pydicom.dcmread(path)
    arr  = ds.pixel_array.astype(np.float32)
    if arr.ndim == 2: arr = arr[np.newaxis]
    lo, hi = np.percentile(arr, 1), np.percentile(arr, 99)
    return np.clip((arr - lo) / (hi - lo + 1e-8), 0, 1)


def load_sheffield_gt(idx):
    """Decode Sheffield segmentation DICOM → (D,H,W) int32 labels 1–37."""
    path = os.path.join(GT_DIR, f'Aug_{idx}_segmentations.dcm')
    ds   = pydicom.dcmread(path)
    raw  = ds.pixel_array.astype(np.float32)
    if raw.ndim == 2: raw = raw[np.newaxis]
    labeled = np.round(raw * 37.0 / 255.0).astype(np.int32)
    labeled[raw == 0] = 0
    return np.clip(labeled, 0, 37)


def build_gt_overlay(seg_arr, alpha=0.5):
    present = {k: v for k, v in SHEFFIELD_GT_LABELS.items() if np.any(seg_arr == k)}
    n       = max(len(present), 1)
    cmap    = plt.colormaps['tab20'].resampled(n)
    seq     = {i: (orig, name) for i, (orig, name) in enumerate(present.items(), 1)}
    rgba    = np.zeros((*seg_arr.shape, 4), dtype=np.float32)
    patches = []
    for i, (orig, name) in seq.items():
        c = (*cmap(i - 1)[:3], alpha)
        rgba[seg_arr == orig] = c
        patches.append(mpatches.Patch(color=c[:3], alpha=0.8, label=name))
    return rgba, patches


def build_nifti_overlay(seg_arr, label_map, alpha=0.5):
    present = {k: v for k, v in label_map.items() if np.any(seg_arr == k)}
    n       = max(len(present), 1)
    cmap    = plt.colormaps['tab20'].resampled(n)
    seq     = {i: (orig, name) for i, (orig, name) in enumerate(present.items(), 1)}
    rgba    = np.zeros((*seg_arr.shape, 4), dtype=np.float32)
    patches = []
    for i, (orig, name) in seq.items():
        c = (*cmap(i - 1)[:3], alpha)
        rgba[seg_arr == orig] = c
        patches.append(mpatches.Patch(color=c[:3], alpha=0.8, label=name))
    return rgba, patches


def build_npz_overlay(data, alpha=0.5):
    names = list(data.files)
    n     = max(len(names), 1)
    cmap  = plt.colormaps['tab20'].resampled(n)
    shape = data[names[0]].shape
    rgba  = np.zeros((*shape, 4), dtype=np.float32)
    patches = []
    for i, name in enumerate(names, 1):
        c = (*cmap(i - 1)[:3], alpha)
        rgba[data[name] > 0] = c
        patches.append(mpatches.Patch(color=c[:3], alpha=0.8, label=name))
    return rgba, patches


def load_seg_overlay(seg_path, cfg):
    if cfg['fmt'] == 'nifti':
        arr = sitk.GetArrayFromImage(sitk.ReadImage(seg_path)).astype(np.int32)
        return build_nifti_overlay(arr, cfg['label_map'])
    else:
        return build_npz_overlay(np.load(seg_path))


def get_stacks(algo_name):
    """Return {Aug_N: seg_path} dict sorted by N."""
    cfg   = AVAILABLE[algo_name]
    files = glob.glob(os.path.join(cfg['seg_dir'], cfg['glob']))
    result = {}
    for f in files:
        m = re.search(r'Aug_(\d+)', f.replace('\\', '/'))
        if m:
            result[f'Aug_{m.group(1)}'] = f
    return dict(sorted(result.items(), key=lambda kv: int(re.search(r'(\d+)', kv[0]).group())))


print('Helpers ready.')

In [ ]:
# ── Widgets ────────────────────────────────────────────────────────────────────

if not AVAILABLE:
    print('No algorithms available yet — run the Lambda notebooks and download results.')
else:
    ALGO_OPTIONS = ['— none —'] + list(AVAILABLE)

    algo1_dd   = Dropdown(options=list(AVAILABLE), description='Algorithm 1:',
                          layout=widgets.Layout(width='420px'))
    algo2_dd   = Dropdown(options=ALGO_OPTIONS, value='— none —',
                          description='Algorithm 2:',
                          layout=widgets.Layout(width='420px'))
    stack_dd   = Dropdown(options=[], description='Stack:',
                          layout=widgets.Layout(width='220px'))
    slice_sl   = IntSlider(min=0, max=1, step=1, value=0, description='Slice:',
                           layout=widgets.Layout(width='600px'))
    show_gt_cb = widgets.Checkbox(value=False, description='Show GT',
                                   indent=False, layout=widgets.Layout(width='110px'))
    out = widgets.Output()

    _cache    = {}
    _gt_cache = {}


    def _idx_from_label(stack_label):
        return re.search(r'(\d+)', stack_label).group()


    def _load(algo_name, stack_label):
        key = (algo_name, stack_label)
        if key not in _cache:
            cfg      = AVAILABLE[algo_name]
            stacks   = get_stacks(algo_name)
            seg_path = stacks[stack_label]
            idx      = _idx_from_label(stack_label)
            img_norm = load_sheffield_image(idx)
            overlay, patches = load_seg_overlay(seg_path, cfg)
            _cache[key] = (img_norm, overlay, patches)
        return _cache[key]


    def _load_gt(stack_label):
        key = stack_label
        if key not in _gt_cache:
            idx     = _idx_from_label(stack_label)
            gt_arr  = load_sheffield_gt(idx)
            _gt_cache[key] = build_gt_overlay(gt_arr)
        return _gt_cache[key]


    def render(algo1, algo2, stack_label, slice_idx, show_gt):
        if not stack_label:
            return
        try:
            img_norm, ov1, patches1 = _load(algo1, stack_label)
        except Exception as e:
            with out:
                out.clear_output(wait=True)
                print(f'Error loading {algo1}: {e}')
            return

        img = img_norm[slice_idx]

        has_algo2 = algo2 != '— none —' and algo2 in AVAILABLE
        ov2, patches2 = None, []
        if has_algo2:
            try:
                _, ov2, patches2 = _load(algo2, stack_label)
            except Exception:
                has_algo2 = False

        gt_ov, gt_patches, gt_err = None, [], None
        if show_gt:
            try:
                gt_ov, gt_patches = _load_gt(stack_label)
            except Exception as e:
                gt_err = str(e)

        panels = [('Greyscale image', None, None)]
        if show_gt:
            panels.append(('Ground truth (37 muscles)',
                            gt_ov, gt_patches if gt_err is None else []))
        panels.append((algo1, ov1, patches1))
        if has_algo2:
            panels.append((algo2, ov2, patches2))
        else:
            lbl = 'Algorithm 2 — select above' if algo2 == '— none —' \
                  else f'{algo2}\n(no matching sample)'
            panels.append((lbl, None, None))

        n_panels  = len(panels)
        fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))
        if n_panels == 1:
            axes = [axes]

        for ax, (title, overlay, patches) in zip(axes, panels):
            ax.imshow(img, cmap='gray', origin='lower')
            if overlay is not None:
                ax.imshow(overlay[slice_idx], origin='lower')
            if patches:
                ax.legend(handles=patches, loc='lower right', fontsize=5,
                          framealpha=0.7, ncol=2)
            color = 'gray' if overlay is None and not patches else 'black'
            ax.set_title(title, fontsize=10, color=color)
            ax.axis('off')

        fig.suptitle(f'{stack_label}  —  slice {slice_idx}', fontsize=11)
        plt.tight_layout()
        with out:
            out.clear_output(wait=True)
            plt.show()
            if gt_err:
                print(f'[GT] {gt_err}')


    def _rerender(*_):
        render(algo1_dd.value, algo2_dd.value, stack_dd.value,
               slice_sl.value, show_gt_cb.value)


    def on_algo1_change(change):
        _cache.clear()
        stacks = get_stacks(change['new'])
        stack_dd.options = list(stacks)
        if stacks:
            stack_dd.value = list(stacks)[0]
            slice_sl.max   = _load(change['new'], stack_dd.value)[0].shape[0] - 1
            slice_sl.value = 0
        _rerender()


    def on_stack_change(change):
        if change['new']:
            img_norm, *_ = _load(algo1_dd.value, change['new'])
            slice_sl.max   = img_norm.shape[0] - 1
            slice_sl.value = 0
        _rerender()


    algo1_dd.observe(on_algo1_change, names='value')
    algo2_dd.observe(lambda _: _rerender(), names='value')
    stack_dd.observe(on_stack_change, names='value')
    slice_sl.observe(lambda _: _rerender(), names='value')
    show_gt_cb.observe(lambda _: _rerender(), names='value')

    # Initial load
    stacks = get_stacks(algo1_dd.value)
    stack_dd.options = list(stacks)
    if stacks:
        stack_dd.value = list(stacks)[0]
        img0, *_ = _load(algo1_dd.value, stack_dd.value)
        slice_sl.max = img0.shape[0] - 1
    render(algo1_dd.value, algo2_dd.value, stack_dd.value, 0, show_gt_cb.value)

    display(VBox([
        HBox([algo1_dd, algo2_dd, show_gt_cb]),
        HBox([stack_dd, slice_sl]),
        out,
    ]))